In [ ]:
# Onyxia GPU compatible torch version
import sys
!{sys.executable} -m pip install torch==2.11.0 torchaudio torchvision \
    --index-url https://download.pytorch.org/whl/cu126 \
    --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached torch-2.11.0%2Bcu126-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchaudio-2.11.0%2Bcu126-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached torchvision-0.26.0%2Bcu126-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-70.2.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-12.9.4-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 94.3 MB/s  0:00:

In [1]:
!pip install sentence-transformers

In [2]:
import pandas as pd
import os
import numpy as np
import re
import string
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TODO reporductibilité du chemin

In [3]:
# ── 1. CHARGER LE CORPUS ──────────────────────────────────────────────────────

#Chemin à adapter
input_dir = "data/"

df = pd.DataFrame()
for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(input_dir, file))
        df = pd.concat([df, temp])

In [4]:
# ── 2. NETTOYAGE ──────────────────────────────────────────────────────────────

df = df[(df["main_g"]=="female") | (df["main_g"]=="male")]
df["text"] = df["text"].apply(lambda x: x.split(". "))
df = df.explode("text")
df["text"] = df["text"].apply(lambda x: re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())
df["text"] = df["text"].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation.replace("'", ""))))
df["g_ath"] = df["audio_file"].apply(lambda x: x.replace(".wav", "").split("-")[-1])
df = df.reset_index(drop=True)


In [5]:
# ── 3. EXPLORER LE VOCABULAIRE RÉEL DU CORPUS ─────────────────────────────────

STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

words = " ".join(df["text"].tolist()).split()
words_filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]
freq = Counter(words_filtered).most_common(50)

print("=== 50 mots les plus fréquents (hors stopwords) ===")
for word, count in freq:
    print(f"  {word}: {count}")

=== 50 mots les plus fréquents (hors stopwords) ===
  'est: 682
  secondes: 671
  tir: 456
  course: 306
  relais: 277
  faut: 247
  temps: 246
  médaille: 237
  vraiment: 234
  tête: 230
  petit: 227
  déjà: 215
  skis: 190
  également: 183
  vite: 182
  français: 176
  piste: 173
  peutêtre: 169
  monde: 167
  l'instant: 155
  voit: 147
  qu'on: 145
  chercher: 145
  olympique: 143
  tour: 134
  aller: 133
  coucher: 125
  train: 117
  sprint: 116
  n'est: 111
  ski: 111
  podium: 111
  pénalité: 109
  dernière: 108
  face: 108
  attention: 104
  titre: 102
  passe: 101
  place: 100
  n'a: 100
  passer: 99
  faute: 98
  l'a: 97
  médailles: 94
  rapport: 94
  qu'elle: 93
  moment: 93
  l'équipe: 92
  fort: 89
  rapide: 88


In [ ]:
# ── 4. CALCULER LES EMBEDDINGS (GPU + cache) ──────────────────────────────────

embeddings_path = "work/PESSD-GBDOC/embeddings.npy"

if os.path.exists(embeddings_path):
    print("\nChargement des embeddings depuis le cache...")
    embeddings = np.load(embeddings_path)
    print(f"Embeddings chargés : {embeddings.shape}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nCalcul des embeddings sur : {device}")

    embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device=device)

    embeddings = embedding_model.encode(
        df["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    np.save(embeddings_path, embeddings)
    print(f"Embeddings sauvegardés dans {embeddings_path}")

df["embedding"] = list(embeddings)
print(f"Corpus : {len(df)} phrases | embeddings : {embeddings.shape}")


In [8]:
# ── 5. DISTANCE COSINUS PAR TERME ─────────────────────────────────────────────

# Adapter cette liste après avoir regardé les 50 mots les plus fréquents
TERMES = ["position", "effort", "finale", "olympique", "tête", "descente", "ski", "marche","médaille","sprint","course"]

results = []

for terme in TERMES:
    mask_F = df["g_ath"].str.upper() == "F"
    mask_H = df["g_ath"].str.upper() == "H"
    mask_terme = df["text"].str.contains(terme, case=False, na=False)

    emb_F = np.array(df[mask_F & mask_terme]["embedding"].tolist())
    emb_H = np.array(df[mask_H & mask_terme]["embedding"].tolist())

    if len(emb_F) == 0 or len(emb_H) == 0:
        results.append({"terme": terme,  "similarite_cosinus": None})
        continue

    centroid_F = emb_F.mean(axis=0, keepdims=True)
    centroid_H = emb_H.mean(axis=0, keepdims=True)

    sim = cosine_similarity(centroid_F, centroid_H)[0][0]
    results.append({"terme": terme, "similarite_cosinus": round(sim, 4)})

# ── 6. RÉSULTATS ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).dropna(subset=["similarite_cosinus"]).sort_values("similarite_cosinus")
print("\n=== Similarité cosinus F vs H par terme ===")
print(results_df.to_string(index=False))


=== Similarité cosinus F vs H par terme ===
    terme  similarite_cosinus
   course              0.4344
   sprint              0.5698
      ski              0.7866
   effort              0.7880
   marche              0.8015
 médaille              0.8099
     tête              0.8175
 descente              0.8317
olympique              0.8557
   finale              0.9127
 position              0.9396
